# 03 - Technical Indicators Generation

## Objective
Generate technical indicators and return-based features for each stock using the raw OHLCV dataset.

## Input
- `Market_Data/raw/raw_stock_data.csv`

## Output
- `Market_Data/features/stock_with_indicators.csv`

## Features Generated
- **Return Features**: Daily Return, Log Return
- **Momentum Indicators**: RSI (14-day), ROC
- **Trend Indicators**: EMA (20-day), SMA (20-day), MACD, MACD Signal
- **Volatility Indicators**: Bollinger Bands, ATR, Rolling Volatility (20/50-day)
- **Volume Indicators**: Volume MA (20-day), OBV

## 1. Import Libraries

In [11]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Technical Analysis library
import ta
from ta.momentum import RSIIndicator, ROCIndicator
from ta.trend import EMAIndicator, SMAIndicator, MACD
from ta.volatility import BollingerBands, AverageTrueRange
from ta.volume import OnBalanceVolumeIndicator

print(f"pandas version: {pd.__version__}")
print("ta library loaded successfully")

pandas version: 2.3.3
ta library loaded successfully


## 2. Load Dataset

In [12]:
# Load raw stock data
df = pd.read_csv('../Market_Data/raw/raw_stock_data.csv')

print(f"Dataset Shape: {df.shape}")
print(f"\nColumns: {df.columns.tolist()}")
print(f"\nData Types:\n{df.dtypes}")
df.head()

Dataset Shape: (70930, 7)

Columns: ['Date', 'Ticker', 'Open', 'High', 'Low', 'Close', 'Volume']

Data Types:
Date       object
Ticker     object
Open      float64
High      float64
Low       float64
Close     float64
Volume      int64
dtype: object


,Date,Ticker,Open,High,Low,Close,Volume
0,2023-01-02,ABB,2685.000000,2713.00,2662.500000,2679.949951,135840
1,2023-01-02,ADANIENT,3870.000000,3874.00,3822.550049,3841.199951,923051
2,2023-01-02,ADANIGREEN,1933.699951,1948.25,1880.000000,1888.699951,695287
3,2023-01-02,ADANIPORTS,823.000000,826.75,816.299988,822.299988,2042294
4,2023-01-02,ADANIPOWER,60.599998,60.98,58.660000,59.630001,8688375


## 3. Data Preprocessing

In [13]:
# Convert Date to datetime
df['Date'] = pd.to_datetime(df['Date'])

# Sort by Ticker and Date (important for time-series operations)
df = df.sort_values(['Ticker', 'Date']).reset_index(drop=True)

print(f"Date range: {df['Date'].min()} to {df['Date'].max()}")
print(f"Number of unique stocks: {df['Ticker'].nunique()}")
print(f"Stocks: {df['Ticker'].unique()[:10]}...")  # Show first 10

Date range: 2023-01-02 00:00:00 to 2025-12-31 00:00:00
Number of unique stocks: 101
Stocks: ['ABB' 'ADANIENSOL' 'ADANIENT' 'ADANIGREEN' 'ADANIPORTS' 'ADANIPOWER'
 'AMBUJACEM' 'APOLLOHOSP' 'ASIANPAINT' 'AXISBANK']...


## 4. Define Technical Indicator Calculation Function

All indicators are calculated per stock to avoid mixing data between different stocks.

In [14]:
def calculate_technical_indicators(group):
    """
    Calculate all technical indicators for a single stock.
    
    Parameters:
    -----------
    group : pd.DataFrame
        DataFrame containing OHLCV data for a single stock
    
    Returns:
    --------
    pd.DataFrame
        DataFrame with all technical indicators added
    """
    df_stock = group.copy()
    
    # Ensure data is sorted by date
    df_stock = df_stock.sort_values('Date').reset_index(drop=True)
    
    # Extract price and volume series
    close = df_stock['Close']
    high = df_stock['High']
    low = df_stock['Low']
    volume = df_stock['Volume']
    
    # ========== RETURN FEATURES ==========
    # Daily Return (percentage change)
    df_stock['Return'] = close.pct_change()
    
    # Log Return
    df_stock['Log_Return'] = np.log(close / close.shift(1))
    
    # ========== MOMENTUM INDICATORS ==========
    # RSI (14-day)
    rsi_indicator = RSIIndicator(close=close, window=14)
    df_stock['RSI'] = rsi_indicator.rsi()
    
    # ROC (Rate of Change) - 10-day default
    roc_indicator = ROCIndicator(close=close, window=10)
    df_stock['ROC'] = roc_indicator.roc()
    
    # ========== TREND INDICATORS ==========
    # EMA (20-day)
    ema_indicator = EMAIndicator(close=close, window=20)
    df_stock['EMA_20'] = ema_indicator.ema_indicator()
    
    # SMA (20-day)
    sma_indicator = SMAIndicator(close=close, window=20)
    df_stock['SMA_20'] = sma_indicator.sma_indicator()
    
    # MACD
    macd_indicator = MACD(close=close, window_slow=26, window_fast=12, window_sign=9)
    df_stock['MACD'] = macd_indicator.macd()
    df_stock['MACD_Signal'] = macd_indicator.macd_signal()
    
    # ========== VOLATILITY INDICATORS ==========
    # Bollinger Bands (20-day, 2 std)
    bb_indicator = BollingerBands(close=close, window=20, window_dev=2)
    df_stock['BB_upper'] = bb_indicator.bollinger_hband()
    df_stock['BB_lower'] = bb_indicator.bollinger_lband()
    
    # ATR (Average True Range) - 14-day
    atr_indicator = AverageTrueRange(high=high, low=low, close=close, window=14)
    df_stock['ATR'] = atr_indicator.average_true_range()
    
    # Rolling Volatility (standard deviation of returns)
    df_stock['Volatility_20'] = df_stock['Return'].rolling(window=20).std()
    df_stock['Volatility_50'] = df_stock['Return'].rolling(window=50).std()
    
    # ========== VOLUME INDICATORS ==========
    # Volume Moving Average (20-day)
    df_stock['Volume_MA_20'] = volume.rolling(window=20).mean()
    
    # On Balance Volume (OBV)
    obv_indicator = OnBalanceVolumeIndicator(close=close, volume=volume)
    df_stock['OBV'] = obv_indicator.on_balance_volume()
    
    return df_stock

## 5. Apply Technical Indicators to All Stocks

In [15]:
print("Calculating technical indicators for all stocks...")
print(f"Total stocks to process: {df['Ticker'].nunique()}")

# Apply indicator calculation function to each stock group
df_with_indicators = df.groupby('Ticker', group_keys=False).apply(calculate_technical_indicators)

# Reset index to ensure clean DataFrame
df_with_indicators = df_with_indicators.reset_index(drop=True)

print(f"\nDataset shape after adding indicators: {df_with_indicators.shape}")
print(f"\nColumns: {df_with_indicators.columns.tolist()}")

Calculating technical indicators for all stocks...
Total stocks to process: 101

Dataset shape after adding indicators: (70930, 22)

Columns: ['Date', 'Ticker', 'Open', 'High', 'Low', 'Close', 'Volume', 'Return', 'Log_Return', 'RSI', 'ROC', 'EMA_20', 'SMA_20', 'MACD', 'MACD_Signal', 'BB_upper', 'BB_lower', 'ATR', 'Volatility_20', 'Volatility_50', 'Volume_MA_20', 'OBV']


## 6. Check Missing Values Before Cleanup

In [16]:
print("Missing values per column BEFORE cleanup:")
print("="*50)
missing_before = df_with_indicators.isnull().sum()
print(missing_before[missing_before > 0])
print(f"\nTotal rows: {len(df_with_indicators)}")

Missing values per column BEFORE cleanup:
Return            101
Log_Return        101
RSI              1313
ROC              1010
EMA_20           1919
SMA_20           1919
MACD             2525
MACD_Signal      3333
BB_upper         1919
BB_lower         1919
Volatility_20    2020
Volatility_50    5035
Volume_MA_20     1919
dtype: int64

Total rows: 70930


## 7. Remove Rows with NaN Values (Due to Rolling Windows)

In [17]:
# The indicator columns that may have NaN due to rolling windows
indicator_columns = [
    'Return', 'Log_Return', 'RSI', 'ROC', 'EMA_20', 'SMA_20',
    'MACD', 'MACD_Signal', 'BB_upper', 'BB_lower', 'ATR',
    'Volatility_20', 'Volatility_50', 'Volume_MA_20', 'OBV'
]

# Count rows before
rows_before = len(df_with_indicators)

# Drop rows with NaN in any indicator column
df_clean = df_with_indicators.dropna(subset=indicator_columns).reset_index(drop=True)

# Count rows after
rows_after = len(df_clean)

print(f"Rows before cleanup: {rows_before:,}")
print(f"Rows after cleanup: {rows_after:,}")
print(f"Rows removed: {rows_before - rows_after:,}")
print(f"Percentage retained: {(rows_after/rows_before)*100:.2f}%")

Rows before cleanup: 70,930
Rows after cleanup: 65,895
Rows removed: 5,035
Percentage retained: 92.90%


## 8. Reorder Columns

In [18]:
# Define final column order as specified
final_columns = [
    'Date', 'Ticker', 'Open', 'High', 'Low', 'Close', 'Volume',
    'Return', 'Log_Return', 'RSI', 'ROC', 'EMA_20', 'SMA_20',
    'MACD', 'MACD_Signal', 'BB_upper', 'BB_lower', 'ATR',
    'Volatility_20', 'Volatility_50', 'Volume_MA_20', 'OBV'
]

# Reorder columns
df_final = df_clean[final_columns].copy()

print(f"Final columns: {df_final.columns.tolist()}")
print(f"Total columns: {len(df_final.columns)}")

Final columns: ['Date', 'Ticker', 'Open', 'High', 'Low', 'Close', 'Volume', 'Return', 'Log_Return', 'RSI', 'ROC', 'EMA_20', 'SMA_20', 'MACD', 'MACD_Signal', 'BB_upper', 'BB_lower', 'ATR', 'Volatility_20', 'Volatility_50', 'Volume_MA_20', 'OBV']
Total columns: 22


## 9. Validation Checks

In [19]:
print("="*60)
print("VALIDATION CHECKS")
print("="*60)

# Dataset shape
print(f"\n1. Dataset Shape: {df_final.shape}")

# Number of unique stocks
print(f"\n2. Number of Unique Stocks: {df_final['Ticker'].nunique()}")

# Missing values per column
print(f"\n3. Missing Values Per Column:")
missing_values = df_final.isnull().sum()
if missing_values.sum() == 0:
    print("   No missing values!")
else:
    print(missing_values[missing_values > 0])

VALIDATION CHECKS

1. Dataset Shape: (65895, 22)

2. Number of Unique Stocks: 100

3. Missing Values Per Column:
   No missing values!


In [20]:
# First 5 rows
print("4. First 5 Rows:")
df_final.head()

4. First 5 Rows:


,Date,Ticker,Open,High,Low,Close,Volume,Return,Log_Return,RSI,...,SMA_20,MACD,MACD_Signal,BB_upper,BB_lower,ATR,Volatility_20,Volatility_50,Volume_MA_20,OBV
0,2023-03-15,ABB,3344.000000,3347.949951,3286.000000,3302.250000,228737,-0.002688,-0.002691,61.089784,...,3245.569983,88.782405,96.142584,3429.555071,3061.584895,85.322652,0.015609,0.016247,348395.80,4383288
1,2023-03-16,ABB,3282.050049,3344.649902,3252.000000,3327.949951,257895,0.007783,0.007752,63.064254,...,3254.487476,85.193424,93.952752,3436.277668,3072.697283,85.846027,0.014939,0.016254,330304.45,4641183
2,2023-03-17,ABB,3347.050049,3364.000000,3286.600098,3312.550049,256724,-0.004627,-0.004638,61.064636,...,3260.694983,80.182196,91.198641,3441.510581,3079.879385,85.242732,0.014855,0.016302,330495.80,4384459
3,2023-03-20,ABB,3300.000000,3329.000000,3266.449951,3298.850098,175472,-0.004136,-0.004144,59.264343,...,3268.829993,74.249390,87.808791,3440.930063,3096.729923,83.621826,0.014298,0.015626,321571.10,4208987
4,2023-03-21,ABB,3299.000000,3432.949951,3299.000000,3412.350098,278420,0.034406,0.033827,67.747821,...,3280.740002,77.809164,85.808866,3457.877748,3103.602257,87.227400,0.015860,0.016174,320222.80,4487407


In [21]:
# Last 5 rows
print("5. Last 5 Rows:")
df_final.tail()

5. Last 5 Rows:


,Date,Ticker,Open,High,Low,Close,Volume,Return,Log_Return,RSI,...,SMA_20,MACD,MACD_Signal,BB_upper,BB_lower,ATR,Volatility_20,Volatility_50,Volume_MA_20,OBV
65890,2025-12-24,ZYDUSLIFE,937.000000,937.000000,916.00,917.799988,745235,-0.010778,-0.010837,41.659282,...,926.614996,-7.981766,-9.303610,944.442575,908.787417,14.554213,0.006367,0.009271,702768.10,98351308
65891,2025-12-26,ZYDUSLIFE,917.849976,920.000000,909.00,911.549988,304792,-0.006810,-0.006833,38.567201,...,925.329996,-8.237246,-9.090338,943.605332,907.054660,14.300340,0.006472,0.009057,679576.75,98046516
65892,2025-12-29,ZYDUSLIFE,912.200012,914.599976,902.25,904.599976,487836,-0.007624,-0.007654,35.418979,...,923.434995,-8.897953,-9.051861,942.052425,904.817564,14.161028,0.006397,0.009100,657972.10,97558680
65893,2025-12-30,ZYDUSLIFE,904.000000,906.000000,896.75,901.450012,721717,-0.003482,-0.003488,34.061853,...,921.684995,-9.565479,-9.154584,941.613579,901.756410,13.810241,0.006325,0.009011,667347.65,96836963
65894,2025-12-31,ZYDUSLIFE,902.000000,915.700012,902.00,914.349976,383442,0.014310,0.014209,43.593743,...,920.267493,-8.950404,-9.113748,937.918145,902.616840,13.841652,0.007057,0.009269,645349.75,97220405


In [22]:
# Additional validation: Check data types
print("6. Data Types:")
print(df_final.dtypes)

6. Data Types:
Date             datetime64[ns]
Ticker                   object
Open                    float64
High                    float64
Low                     float64
Close                   float64
Volume                    int64
Return                  float64
Log_Return              float64
RSI                     float64
ROC                     float64
EMA_20                  float64
SMA_20                  float64
MACD                    float64
MACD_Signal             float64
BB_upper                float64
BB_lower                float64
ATR                     float64
Volatility_20           float64
Volatility_50           float64
Volume_MA_20            float64
OBV                       int64
dtype: object


In [23]:
# Statistical summary of indicators
print("7. Statistical Summary of Indicators:")
df_final[['Return', 'Log_Return', 'RSI', 'ROC', 'MACD', 'ATR', 'Volatility_20', 'OBV']].describe()

7. Statistical Summary of Indicators:


,Return,Log_Return,RSI,ROC,MACD,ATR,Volatility_20,OBV
count,65895.000000,65895.000000,65895.000000,65895.000000,65895.000000,65895.000000,65895.000000,6.589500e+04
mean,0.001138,0.000966,53.448457,1.105595,13.507781,61.542841,0.016861,4.491612e+08
std,0.018550,0.018524,12.177575,5.854497,98.161088,109.145288,0.008033,1.068763e+09
min,-0.251944,-0.290277,12.225743,-55.051236,-1369.919289,0.602736,0.003169,-6.681363e+08
25%,-0.008146,-0.008179,44.819737,-2.314447,-4.703507,11.333041,0.011696,1.644065e+07
50%,0.000493,0.000493,53.464015,0.779662,3.205987,26.112956,0.015120,8.108813e+07
75%,0.009603,0.009558,61.977839,4.047363,19.705496,62.281342,0.019817,3.837947e+08
max,0.256015,0.227944,92.579422,77.609492,1744.982205,1117.272579,0.086057,9.842793e+09


In [24]:
# Verify date range
print("8. Date Range:")
print(f"   Start Date: {df_final['Date'].min()}")
print(f"   End Date: {df_final['Date'].max()}")

# Check rows per stock (should be consistent after cleanup)
print("\n9. Rows per Stock (sample):")
rows_per_stock = df_final.groupby('Ticker').size()
print(f"   Min rows per stock: {rows_per_stock.min()}")
print(f"   Max rows per stock: {rows_per_stock.max()}")
print(f"   Mean rows per stock: {rows_per_stock.mean():.1f}")

8. Date Range:
   Start Date: 2023-03-15 00:00:00
   End Date: 2025-12-31 00:00:00

9. Rows per Stock (sample):
   Min rows per stock: 3
   Max rows per stock: 690
   Mean rows per stock: 659.0


## 10. Save Dataset

In [25]:
# Save to features folder
output_path = '../Market_Data/features/stock_with_indicators.csv'

df_final.to_csv(output_path, index=False)

print(f"Dataset saved to: {output_path}")
print(f"Final shape: {df_final.shape}")
print(f"File size: {pd.read_csv(output_path).memory_usage(deep=True).sum() / 1024**2:.2f} MB (estimated)")

Dataset saved to: ../Market_Data/features/stock_with_indicators.csv
Final shape: (65895, 22)
File size: 18.30 MB (estimated)


## 11. Summary

### Features Generated:

| Category | Feature | Description |
|----------|---------|-------------|
| **Returns** | Return | Daily percentage change |
| | Log_Return | Logarithmic return |
| **Momentum** | RSI | Relative Strength Index (14-day) |
| | ROC | Rate of Change (10-day) |
| **Trend** | EMA_20 | Exponential Moving Average (20-day) |
| | SMA_20 | Simple Moving Average (20-day) |
| | MACD | Moving Average Convergence Divergence |
| | MACD_Signal | MACD Signal Line |
| **Volatility** | BB_upper | Bollinger Band Upper |
| | BB_lower | Bollinger Band Lower |
| | ATR | Average True Range (14-day) |
| | Volatility_20 | 20-day rolling std of returns |
| | Volatility_50 | 50-day rolling std of returns |
| **Volume** | Volume_MA_20 | Volume Moving Average (20-day) |
| | OBV | On Balance Volume |

### Next Steps:
- Proceed to `04_global_market_data.ipynb` to download global market proxy data

In [26]:
print("\n" + "="*60)
print("NOTEBOOK COMPLETED SUCCESSFULLY!")
print("="*60)
print(f"\nOutput: Market_Data/features/stock_with_indicators.csv")
print(f"Shape: {df_final.shape[0]:,} rows x {df_final.shape[1]} columns")
print(f"Stocks: {df_final['Ticker'].nunique()}")


NOTEBOOK COMPLETED SUCCESSFULLY!

Output: Market_Data/features/stock_with_indicators.csv
Shape: 65,895 rows x 22 columns
Stocks: 100
